In [ ]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

In [ ]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [ ]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [ ]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Payload with Application Message

### CQ3.01u - Which Application Message encodes which SOSA Activity?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?message ?activity ?member ?FOI ?property WHERE {
  ?message a mqtt:ApplicationMessage ;
          ?applicationMessageEncodesActivity ?activity .
  ?activity a mqtt:Activity .
  OPTIONAL {?activity sosa:hasFeatureOfInterest ?FOI}
  OPTIONAL {?activity sosa:hasMember ?member}
  OPTIONAL {?member sosa:observedProperty ?property}
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.02u - Which fields are encoded in the Application Message?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?fields WHERE {
  ?message a mqtt:ApplicationMessage ;
          mqtt:hasEncodedFields ?fields .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.03u - What content, content type, encoding and delimiter does an Application Message have?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?content ?type ?encoding ?delimiter WHERE {
  ?message a mqtt:ApplicationMessage ;
  OPTIONAL { ?message mqtt:hasApplicationContent ?content }
  OPTIONAL { ?message mqtt:hasApplicationContentType ?type }
  OPTIONAL { ?message mqtt:hasApplicationEncoding ?encoding }
  OPTIONAL { ?message mqtt:hasApplicationContentDelimiter ?delimiter }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.04u - Who has send which Application Messages?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?payload ?packet ?client ?broker WHERE {
  ?message a mqtt:ApplicationMessage ;
           mqtt:isApplicationMessageOf ?payload .
  ?payload mqtt:isPublishPayloadOf ?packet .
  OPTIONAL { ?client mqtt:sendsControlPacket ?packet . ?client a mqtt:Client }
  OPTIONAL { ?broker mqtt:sendsControlPacket ?packet . ?broker a mqtt:Broker }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.05u - Which MQTT Publish packets carry Observations for a specific SOSA FeatureOfInterest or Property?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?packet ?message ?observationCollection ?foi ?property WHERE {
  ?packet a mqtt:PublishPacket ;
          mqtt:hasPublishPayload ?payload .
  ?payload a mqtt:Payload ;
          mqtt:hasApplicationMessage ?message .
  ?message a mqtt:ApplicationMessage ;       
          mqtt:applicationMessageEncodesObservationCollection ?observationCollection .
  ?activity a mqtt:Activity .
  OPTIONAL { ?activity sosa:hasFeatureOfInterest ?foi }
  OPTIONAL { 
    ?observationCollection sosa:hasMember ?observation .
    ?observation a sosa:Observation ;
                 sosa:observedProperty ?property .
  }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.06u - Which MQTT SUBSCRIBE packets request QoS=2 and for which filters?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?entry ?filter WHERE {
  ?packet a mqtt:SubscribePacket ;
          mqtt:hasSubscribePayload ?payload .
  ?payload mqtt:hasSubscriptions ?entry .
  ?entry a mqtt:SubscriptionEntry .
  OPTIONAL { ?entry mqtt:hasQoSLevel 2 }
  OPTIONAL { ?entry mqtt:hasTopicFilter ?filter }

}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.07u - Which MQTT UNSUBSCRIBE packets remove all filters under a certain path (e.g. 'factory/centering/#')?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?filter WHERE {
  ?packet a mqtt:UnsubscribePacket ;
          mqtt:hasUnsubscribePayload ?payload .
  ?payload mqtt:hasTopicFilter ?filter .
  FILTER(STRSTARTS(?filter, "factory/centering/"))
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.08u - Which MQTT SUBSCRIBE payloads contain which SubscriptionEntries?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?payload ?entry WHERE {
  ?payload a mqtt:SubscribePayload ;
           mqtt:hasSubscriptions ?entry .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.09u - Which MQTT Topic Filters (with QoS) are requested by a SubscriptionEntry?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?entry ?filter ?qos WHERE {
  ?entry a mqtt:SubscriptionEntry ;
    OPTIONAL { ?entry mqtt:hasTopicFilter ?filter }  
    OPTIONAL { ?entry mqtt:hasQoSLevel ?qos }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ3.10u - Which MQTT UNSUBSCRIBE payloads remove which MQTT TopicFilters?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?payload ?filter WHERE {
  ?payload a mqtt:UnsubscribePayload ;
           mqtt:hasTopicFilter ?filter .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short